# HPDM097 Assignment 2

## Iteration 3: Rehab ward (Gemini)

This is the 3rd iteration to the code. A rehab ward was added to the patient pathway with its own length of stay (LOS) logic and transfer matrix to either ESD or 'Other'. Additionally a separate stream of patients which are direct transfers from other locations to the rehab ward has been added and follow the safe pathway as a patient admitted from the acute ward.

Prompt and response: https://gemini.google.com/share/9633688a34a2

## Prompt:
Before that, let's complete the rest of the logic diagram, again using the iterative approach and recreating the code above but now adding to the patient pathway.

Patients who were transferred to the rehab ward can then be transferred to either ESD (as mentioned previously) or Other (as mentioned previously) with its own LOS logic within the rehab ward and transfer matrix:

Note that there are no acute stroke (mortality) patients in the rehab ward.
There are 12 rehab beds as a resource.

The rehab ward length of stay (LOS) follows a log-normal distribution with the following parameters in days:
Acute stroke (No ESD) - mean: 28.4, stdev: 27.2
Acute stroke (ESD) - mean: 30.3, stdev: 23.1
TIA - mean: 18.7, stdev: 23.5
Complex neurological - mean: 27.6, stdev: 28.4
Other - mean: 16.1, stdev: 14.1

Here is the transfer matrix from the rehab ward:
Stroke patients: 40% to ESD, 60% to Other
TIA patients: 0% to ESD, 100% to Other
Complex neurological patients: 9% to ESD, 91% to Other
Other patients: 13% to ESD, 88% to Other

Additionally, the rehab ward can also receive a new admission/transfer from elsewhere (different hospital etc.). The inter-arrival mean times in days for this subset of direct transfers to the rehab ward are as follows:
Stroke - mean: 21.8 days
TIA - (no patients via this route)
Complex neurological - mean: 31.7 days
Other - mean: 28.6 days

These subset of patients enter the pool and use the same resource as those admitted from the acute stroke unit and follow the same process as above to either ESD or Other

## Response:
This is an excellent progression for our model. Adding a second stage (the rehabilitation ward) and a secondary arrival stream (direct transfers) turns this into a true multi-node network simulation.

A quick note before we dive in: I noticed the Rehab transfer probabilities for "Other" patients sum to 101% (13% ESD + 88% Other). I have gently corrected this in the code to 13% ESD and 87% Other so the mathematical probabilities sum to exactly 1.0, which numpy requires.

I've split the patient flow logic into an acute_process and a rehab_process to keep the code clean and modular. Direct rehab admissions will skip the acute process entirely and go straight to the rehab process, while acute admissions routed to rehab will naturally chain from one process to the next.

Here is the updated code for this iteration:

In [1]:
import simpy
import numpy as np

# ---------------------------------------------------------
# 2. Add a trace utility function as a debug print switch
# ---------------------------------------------------------
TRACE = False

def trace(msg):
    """Utility function to print debug messages if TRACE is True."""
    if TRACE:
        print(msg)

# ---------------------------------------------------------
# 1. Build and sample from distribution classes
# ---------------------------------------------------------
class ExponentialDistribution:
    """Class to sample from an exponential distribution for arrivals."""
    def __init__(self, mean, random_seed=None):
        self.mean = mean
        self.rng = np.random.default_rng(seed=random_seed)
        
    def sample(self):
        return self.rng.exponential(scale=self.mean)

class LognormalDistribution:
    """Class to sample from a lognormal distribution for length of stay."""
    def __init__(self, sample_mean, sample_stdev, random_seed=None):
        # Mathematically required conversion from sample stats to lognormal parameters
        variance = sample_stdev ** 2
        self.mu = np.log((sample_mean ** 2) / np.sqrt(variance + sample_mean ** 2))
        self.sigma = np.sqrt(np.log(1 + (variance / sample_mean ** 2)))
        self.rng = np.random.default_rng(seed=random_seed)
        
    def sample(self):
        return self.rng.lognormal(mean=self.mu, sigma=self.sigma)

# ---------------------------------------------------------
# 3. Create a parameter container class called Scenario
# ---------------------------------------------------------
# 6. Input and specify the model parameters as above
# ---------------------------------------------------------
class Scenario:
    """Contains all parameters and configurations for the simulation."""
    def __init__(self):
        self.random_seed = 42
        
        # Resource Capacities
        self.acute_beds_capacity = 10
        self.rehab_beds_capacity = 12
        
        # Mean inter-arrival times (in days)
        self.acute_inter_arrival_means = {
            'Acute stroke': 1.2,
            'TIA': 9.3,
            'Complex neurological': 3.6,
            'Other': 3.2
        }
        
        # Direct transfers to Rehab mean inter-arrival times (in days)
        self.rehab_inter_arrival_means = {
            'Acute stroke': 21.8,
            'Complex neurological': 31.7,
            'Other': 28.6
            # TIA has no direct admissions
        }
        
        # Transfer Matrices
        self.acute_destinations = ['Rehab', 'ESD', 'Other']
        self.acute_transfer_matrix = {
            'Acute stroke': [0.24, 0.13, 0.63],
            'TIA': [0.01, 0.01, 0.98],
            'Complex neurological': [0.11, 0.05, 0.84],
            'Other': [0.05, 0.10, 0.85]
        }
        
        self.rehab_destinations = ['ESD', 'Other']
        self.rehab_transfer_matrix = {
            'Acute stroke': [0.40, 0.60],
            'TIA': [0.00, 1.00],
            'Complex neurological': [0.09, 0.91],
            'Other': [0.13, 0.87] # Adjusted 88% to 87% to equal 100%
        }
        
        # Acute Length of Stay Parameters (sample mean, sample stdev)
        self.acute_los_params = {
            'Acute stroke': {'Rehab': (7.4, 8.6), 'ESD': (4.6, 4.8), 'Other': (7.4, 8.6)},
            'TIA': {dest: (1.8, 2.3) for dest in self.acute_destinations},
            'Complex neurological': {dest: (4.0, 5.0) for dest in self.acute_destinations},
            'Other': {dest: (3.8, 5.2) for dest in self.acute_destinations}
        }
        
        # Rehab Length of Stay Parameters (sample mean, sample stdev)
        self.rehab_los_params = {
            'Acute stroke': {'ESD': (30.3, 23.1), 'Other': (28.4, 27.2)},
            'TIA': {dest: (18.7, 23.5) for dest in self.rehab_destinations},
            'Complex neurological': {dest: (27.6, 28.4) for dest in self.rehab_destinations},
            'Other': {dest: (16.1, 14.1) for dest in self.rehab_destinations}
        }

# ---------------------------------------------------------
# 4. Create a patient class called AcutePatient
# ---------------------------------------------------------
class AcutePatient:
    """Represents an individual patient flowing through the system."""
    def __init__(self, p_id, p_type, arrival_time, source="Acute"):
        self.id = p_id
        self.type = p_type
        self.arrival_time = arrival_time
        self.source = source  # Tracks if they came via Acute or Direct to Rehab
        
        # Acute Attributes
        self.acute_destination = None
        self.acute_bed_request_time = None
        self.acute_bed_acquire_time = None
        self.acute_length_of_stay = None
        self.acute_discharge_time = None
        
        # Rehab Attributes
        self.rehab_destination = None
        self.rehab_bed_request_time = None
        self.rehab_bed_acquire_time = None
        self.rehab_length_of_stay = None
        self.rehab_discharge_time = None

# ---------------------------------------------------------
# 5. Create a model class called StrokeUnit
# ---------------------------------------------------------
class StrokeUnit:
    """The main simulation model encapsulating the environment and logic."""
    def __init__(self, env, scenario):
        self.env = env
        self.scenario = scenario
        self.patient_counter = 0
        
        # Initialize Resources
        self.acute_beds = simpy.Resource(env, capacity=self.scenario.acute_beds_capacity)
        self.rehab_beds = simpy.Resource(env, capacity=self.scenario.rehab_beds_capacity)
        
        # Initialize RNGs for routing
        self.acute_routing_rng = np.random.default_rng(seed=self.scenario.random_seed + 100)
        self.rehab_routing_rng = np.random.default_rng(seed=self.scenario.random_seed + 200)
        
        # Setup Distribution Dictionaries
        self.acute_arrival_dists = {}
        self.rehab_arrival_dists = {}
        self.acute_los_dists = {p_type: {} for p_type in self.scenario.acute_inter_arrival_means.keys()}
        self.rehab_los_dists = {p_type: {} for p_type in self.scenario.acute_inter_arrival_means.keys()}
        
        dist_seed = self.scenario.random_seed + 1000
        
        # 1. Setup Acute Arrivals & LOS
        for p_type, mean_iat in self.scenario.acute_inter_arrival_means.items():
            self.acute_arrival_dists[p_type] = ExponentialDistribution(mean=mean_iat, random_seed=dist_seed)
            dist_seed += 1
            for dest in self.scenario.acute_destinations:
                mean, stdev = self.scenario.acute_los_params[p_type][dest]
                self.acute_los_dists[p_type][dest] = LognormalDistribution(mean, stdev, random_seed=dist_seed)
                dist_seed += 1
                
        # 2. Setup Direct Rehab Arrivals & LOS
        for p_type, mean_iat in self.scenario.rehab_inter_arrival_means.items():
            self.rehab_arrival_dists[p_type] = ExponentialDistribution(mean=mean_iat, random_seed=dist_seed)
            dist_seed += 1
            
        for p_type in self.scenario.acute_inter_arrival_means.keys(): # All types might end up in rehab
            for dest in self.scenario.rehab_destinations:
                mean, stdev = self.scenario.rehab_los_params[p_type][dest]
                self.rehab_los_dists[p_type][dest] = LognormalDistribution(mean, stdev, random_seed=dist_seed)
                dist_seed += 1

    def acute_arrivals_generator(self, patient_type):
        """Generates patient arrivals hitting the front door of the acute unit."""
        while True:
            yield self.env.timeout(self.acute_arrival_dists[patient_type].sample())
            self.patient_counter += 1
            patient = AcutePatient(self.patient_counter, patient_type, self.env.now, "Acute")
            trace(f"Day {self.env.now:06.2f} | Patient {patient.id:04d} ({patient.type}) arrived at ACUTE.")
            self.env.process(self.acute_process(patient))

    def rehab_direct_arrivals_generator(self, patient_type):
        """Generates patients who transfer directly into the rehab ward from elsewhere."""
        while True:
            yield self.env.timeout(self.rehab_arrival_dists[patient_type].sample())
            self.patient_counter += 1
            patient = AcutePatient(self.patient_counter, patient_type, self.env.now, "Direct Rehab")
            trace(f"Day {self.env.now:06.2f} | Patient {patient.id:04d} ({patient.type}) arrived directly at REHAB.")
            self.env.process(self.rehab_process(patient))

    def acute_process(self, patient):
        """Handles the flow through the acute stroke unit."""
        # Determine acute discharge destination
        probs = self.scenario.acute_transfer_matrix[patient.type]
        patient.acute_destination = self.acute_routing_rng.choice(self.scenario.acute_destinations, p=probs)
        
        patient.acute_bed_request_time = self.env.now
        
        with self.acute_beds.request() as req:
            yield req
            
            patient.acute_bed_acquire_time = self.env.now
            los = self.acute_los_dists[patient.type][patient.acute_destination].sample()
            patient.acute_length_of_stay = los
            
            yield self.env.timeout(los)
            
            patient.acute_discharge_time = self.env.now
            trace(f"Day {self.env.now:06.2f} | Patient {patient.id:04d} ({patient.type}) left ACUTE for {patient.acute_destination}.")
            
            # If routed to Rehab, trigger the rehab process immediately
            if patient.acute_destination == 'Rehab':
                self.env.process(self.rehab_process(patient))

    def rehab_process(self, patient):
        """Handles the flow through the rehabilitation unit."""
        # Determine rehab discharge destination
        probs = self.scenario.rehab_transfer_matrix[patient.type]
        patient.rehab_destination = self.rehab_routing_rng.choice(self.scenario.rehab_destinations, p=probs)
        
        patient.rehab_bed_request_time = self.env.now
        
        with self.rehab_beds.request() as req:
            yield req
            
            patient.rehab_bed_acquire_time = self.env.now
            wait_time = patient.rehab_bed_acquire_time - patient.rehab_bed_request_time
            trace(f"Day {self.env.now:06.2f} | Patient {patient.id:04d} ({patient.type}) secured a REHAB bed. Wait time: {wait_time:.2f} days.")
            
            los = self.rehab_los_dists[patient.type][patient.rehab_destination].sample()
            patient.rehab_length_of_stay = los
            
            yield self.env.timeout(los)
            
            patient.rehab_discharge_time = self.env.now
            trace(f"Day {self.env.now:06.2f} | Patient {patient.id:04d} ({patient.type}) left REHAB for {patient.rehab_destination}.")

    def run_simulation(self):
        """Starts all arrival generators."""
        for p_type in self.scenario.acute_inter_arrival_means.keys():
            self.env.process(self.acute_arrivals_generator(p_type))
            
        for p_type in self.scenario.rehab_inter_arrival_means.keys():
            self.env.process(self.rehab_direct_arrivals_generator(p_type))

# =========================================================
# Execution / Testing Block
# =========================================================
if __name__ == "__main__":
    TRACE = True  
    print("--- Starting Full Stroke & Rehab Service Simulation ---")
    
    scenario = Scenario()
    env = simpy.Environment()
    model = StrokeUnit(env, scenario)
    model.run_simulation()
    
    # Run for 60 days to allow for longer Rehab stays to process
    env.run(until=60)
    print(f"\nSimulation complete. Total arrivals processed: {model.patient_counter}")

--- Starting Full Stroke & Rehab Service Simulation ---
Day 000.03 | Patient 0001 (Other) arrived at ACUTE.
Day 000.65 | Patient 0002 (Acute stroke) arrived directly at REHAB.
Day 000.65 | Patient 0002 (Acute stroke) secured a REHAB bed. Wait time: 0.00 days.
Day 001.47 | Patient 0003 (Acute stroke) arrived at ACUTE.
Day 001.75 | Patient 0004 (TIA) arrived at ACUTE.
Day 001.88 | Patient 0005 (Acute stroke) arrived at ACUTE.
Day 002.25 | Patient 0006 (Complex neurological) arrived at ACUTE.
Day 002.36 | Patient 0007 (Acute stroke) arrived directly at REHAB.
Day 002.36 | Patient 0007 (Acute stroke) secured a REHAB bed. Wait time: 0.00 days.
Day 003.03 | Patient 0004 (TIA) left ACUTE for Other.
Day 003.14 | Patient 0005 (Acute stroke) left ACUTE for Other.
Day 003.37 | Patient 0003 (Acute stroke) left ACUTE for Rehab.
Day 003.37 | Patient 0003 (Acute stroke) secured a REHAB bed. Wait time: 0.00 days.
Day 004.06 | Patient 0008 (Complex neurological) arrived at ACUTE.
Day 005.15 | Patient 0